# UCMCTrack validation baseline and bounded tuning

Select METHOD after smoke comparison. Use one shared tracker configuration for all four classes and all twelve validation sequences. Test is not used for tuning.


## Environment

Set UA_DETRAC_PROJECT, UA_DETRAC_DATASET, UA_DETRAC_DETECTIONS and UA_DETRAC_CALIBRATION to override paths. Use local Colab storage for repeated image reads.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import json

# Mount Drive on Colab; local kernels find the repository from parent folders.
try:
    from google.colab import drive, output as colab_output
except ImportError:
    pass
else:
    drive.mount('/content/drive')
    colab_output.enable_custom_widget_manager()

project = os.environ.get('UA_DETRAC_PROJECT')
candidates = ([Path(project)] if project else [Path.cwd(), *Path.cwd().parents,
    Path('/content/drive/MyDrive/CS331/CS331-Vehicle-Tracking')])
ROOT = next((p.resolve() for p in candidates
             if (p / 'scripts/tracking_pipeline.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Set UA_DETRAC_PROJECT to the repository directory.')
sys.path.insert(0, str(ROOT))
local_dataset = Path('/content/ua_detrac/dataset_v1.1')
default_dataset = local_dataset if local_dataset.is_dir() else ROOT / 'outputs/ua_detrac/dataset_v1.1'
DATASET = Path(os.environ.get('UA_DETRAC_DATASET', os.environ.get('UA_DETRAC_OUTPUT', default_dataset)))
DETECTIONS = Path(os.environ.get('UA_DETRAC_DETECTIONS', ROOT / 'outputs/detections_full_yolo26l_E17'))
CALIBRATION = Path(os.environ.get('UA_DETRAC_CALIBRATION', ROOT / 'outputs/tracking/calibration/registry.json'))
RUNS = ROOT / 'outputs/tracking'
METHOD = 'homography'


def run_command(action, output=None, method=None, sequences=None, resume=False, extra=()):
    command = [sys.executable, str(ROOT / 'scripts/tracking_pipeline.py'), action,
        '--dataset', str(DATASET), '--detections', str(DETECTIONS),
        '--calibration', str(CALIBRATION), '--upstream', str(ROOT / 'UCMC/UCMCTrack')]
    if output is not None:
        command += ['--output', str(output)]
    if method is not None:
        command += ['--method', method]
    if sequences:
        command += ['--sequences', *sequences]
    if resume:
        command += ['--resume']
    command += list(extra)
    print('Command:', command)
    return subprocess.run(command, cwd=ROOT, check=True)

print('Dataset:', DATASET)
print('Detections:', DETECTIONS)
print('Calibration:', CALIBRATION)

# Apply the compatibility patch after initializing the pinned dependency.
subprocess.run([sys.executable, str(ROOT / 'scripts/setup_ucmc.py')], cwd=ROOT, check=True)


## Dependencies

Skip the next cell when using the prepared tracking environment.


In [ ]:
# Run once in a new environment. Local users should select the .venv-tracking kernel.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                str(ROOT / 'requirements-tracking.txt')], cwd=ROOT, check=True)


## Camera preflight

Select the chosen method in the next cell. The runner fingerprints reviewed calibration and refuses incompatible resume inputs.


In [ ]:
METHOD = 'homography'  # Set to the selected smoke method.
run_command('status', method=METHOD)


## Baseline and evaluation

Run the first scheduled trial in the same tuning directory, so the full schedule reuses this baseline. Baseline: confidence 0.10, high score 0.60, a1=a2=10, wx=wy=0.1, vmax=0.5, max age 30, 25 FPS. HP, interpolation and CMC are disabled.


In [ ]:
TUNING_OUTPUT = RUNS / 'val_tuning'
run_command('tune', output=TUNING_OUTPUT, method=METHOD, resume=True,
            extra=['--max-trials', '1'])


## Tune on validation

At most 15 scheduled trials: baseline, nine one-factor variants, a combined configuration, and four ablations. Duplicate configurations reuse a matching run. Ranking is HOTA, IDF1, fewer IDS, then runtime. Scores are percentages; CLEAR/Identity IoU=0.7 and HOTA thresholds=0.05...0.95. Calibration and protocol remain frozen.


In [ ]:
run_command('tune', output=TUNING_OUTPUT, method=METHOD, resume=True,
            extra=['--max-trials', '15'])


In [ ]:
import pandas as pd
display(pd.read_csv(TUNING_OUTPUT / 'leaderboard.csv'))
best = json.loads((TUNING_OUTPUT / 'best_config.json').read_text())
print(json.dumps(best, indent=2))
# These are tuned development-set scores. Freeze configuration before any final test run.
